# Deep Past Challenge — ByT5 Akkadian→English Translation

Submission notebook for the [Deep Past Initiative: Machine Translation](https://www.kaggle.com/competitions/deep-past-initiative-machine-translation) competition.

**Approach**: Fine-tuned `google/byt5-small` (300M params) on preprocessed training data.

**Constraints**: Kaggle GPU (16GB VRAM), no internet, ≤9 hours runtime.

In [ ]:
import gc
import os
import re
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Preprocessing

In [ ]:
# ── Preprocessing (inline — must stay in sync with src/preprocess.py) ─────────

SUBSCRIPT_MAP = str.maketrans('₀₁₂₃₄₅₆₇₈₉ₓ', '0123456789x')
ACCENT_MAP = {'á': 'a2', 'à': 'a3', 'é': 'e2', 'è': 'e3',
              'í': 'i2', 'ì': 'i3', 'ú': 'u2', 'ù': 'u3'}
H_MAP = {'Ḫ': 'H', 'ḫ': 'h'}


def normalize_h(text):
    for src, dst in H_MAP.items():
        text = text.replace(src, dst)
    return text


def normalize_accents(text):
    for src, dst in ACCENT_MAP.items():
        text = text.replace(src, dst)
    return text


def normalize_subscripts(text):
    return text.translate(SUBSCRIPT_MAP)


def normalize_gaps(text):
    text = re.sub(r'\[\s*…\s*…?\s*\]', ' <big_gap> ', text)
    text = re.sub(r'\[x+\]', ' <gap> ', text)
    text = text.replace('…', ' <big_gap> ')
    text = re.sub(r'\bx{2,}\b', ' <gap> ', text)
    return text


def remove_scribal_notations(text):
    text = re.sub(r'<<.*?>>', '', text)
    text = re.sub(r'<([^<>]*)>', r'\1', text)
    text = text.replace('˹', '').replace('˺', '')
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    text = re.sub(r'(?<!\d)[!?]', '', text)
    text = re.sub(r'(?<!\d)/(?!\d)', '', text)
    text = re.sub(r'(?<!\d):(?!\d)', ' ', text)
    return text


def clean_transliteration(text):
    if not isinstance(text, str):
        return ''
    text = normalize_gaps(text)
    text = remove_scribal_notations(text)
    text = normalize_h(text)
    text = normalize_accents(text)
    text = normalize_subscripts(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def postprocess_prediction(text):
    if not isinstance(text, str):
        return ''
    text = normalize_h(text)
    text = normalize_subscripts(text)
    text = remove_scribal_notations(text)
    text = re.sub(r'\b(\w+)(\s+\1)+\b', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print('Preprocessing functions ready.')

## Load Model

In [ ]:
# ── Load fine-tuned model ─────────────────────────────────────────────
# Model uploaded as Kaggle Dataset — update dataset_sources in kernel-metadata.json
MODEL_PATH = '/kaggle/input/byt5-akkadian-finetuned/best'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH, torch_dtype=torch.float32)
model = model.to(device)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')
if device.type == 'cuda':
    print(f'GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## Load & Preprocess Test Data

In [ ]:
# ── Load test data ───────────────────────────────────────────────────
test_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv')
print(f'Test samples: {len(test_df)}')
print(f'Columns: {list(test_df.columns)}')
print(test_df.head())

# Preprocess transliterations
test_df['transliteration_clean'] = test_df['transliteration'].apply(clean_transliteration)

PREFIX = 'translate Akkadian to English: '
texts = [PREFIX + t for t in test_df['transliteration_clean'].tolist()]
print(f'\nPrepared {len(texts)} inputs for inference')

## Generate Translations

In [ ]:
# ── Inference (tuned for 16GB Kaggle GPU) ──────────────────────────────
# ByT5-small is ~1.2GB in fp32. With beam search (8 beams) and 1024-length
# byte sequences, peak memory ~8-10GB. Batch size 2 keeps us safely under 16GB.
BATCH_SIZE = 2          # Conservative for 16GB — beam search is memory-hungry
NUM_BEAMS = 8
MAX_NEW_TOKENS = 512    # Test data is sentence-level, shorter than training docs
MAX_INPUT_LENGTH = 1024 # Must match training max_length
LENGTH_PENALTY = 1.3

predictions = []
start = time.time()

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Translating'):
    batch_texts = texts[i:i + BATCH_SIZE]
    inputs = tokenizer(
        batch_texts,
        max_length=MAX_INPUT_LENGTH,
        padding=True,
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            length_penalty=LENGTH_PENALTY,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(decoded)

    # Free intermediate tensors to keep memory stable
    del inputs, outputs
    if device.type == 'cuda' and i % 50 == 0:
        torch.cuda.empty_cache()

elapsed = time.time() - start
print(f'\nGenerated {len(predictions)} translations in {elapsed:.1f}s ({elapsed/max(len(texts),1):.2f}s/sample)')
if device.type == 'cuda':
    print(f'Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

## Post-process & Write Submission

In [ ]:
# ── Post-process & write submission ───────────────────────────────────
predictions = [postprocess_prediction(p) for p in predictions]

# Ensure no empty predictions (Kaggle rejects empty strings)
predictions = [p if p.strip() else '<gap>' for p in predictions]

submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': predictions,
})

# Validate submission format
assert list(submission.columns) == ['id', 'translation'], f"Bad columns: {list(submission.columns)}"
assert len(submission) == len(test_df), f"Row count mismatch: {len(submission)} vs {len(test_df)}"
assert submission['translation'].isna().sum() == 0, "Found NaN translations"

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Submission saved: {len(submission)} rows')
print(f'Output file: /kaggle/working/submission.csv')
print(f'\nSubmission head:')
print(submission.head(10))

# Show examples
for i in range(min(5, len(predictions))):
    print(f'\n--- Sample {i} ---')
    print(f'Input:  {test_df.iloc[i]["transliteration"][:150]}')
    print(f'Output: {predictions[i][:150]}')